# AI Coding Agents in AI and Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/konstantin-schekotihin/dl_course/blob/master/AI-ML/09_ml_agents.ipynb)


In [ ]:
# Setup & configuration (skip in presentation slides)
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics as mt


# Load course helper utilities (local file or direct GitHub fetch for standalone Colab)
try:
    import helpers as hp
except ImportError:
    import urllib.request
    _url = "https://raw.githubusercontent.com/konstantin-schekotihin/dl_course/master/AI-ML/helpers.py"
    urllib.request.urlretrieve(_url, "helpers.py")
    import helpers as hp


# AI Coding Agents: Leverage and Responsibility

- Modern AI coding assistants (**Antigravity / Agy, GitHub Copilot, Claude Code**) accelerate exploratory data analysis and code generation.
- **The Course GenAI Policy** (see `modalities.md`):
  - **Transparency**: You are permitted to use AI coding tools, provided you explicitly disclose and attribute them in your final report.
  - **Defensibility (60% Oral Exam)**: An AI assistant cannot sit your exam. You must personally defend and explain every mathematical principle, line of code, and hyperparameter choice in your project.
  - **Analytical Mastery**: The machine assists with typing code; **you** are the machine learning engineer responsible for problem formulation, validation, and conclusions!


## Structured Prompting: Guiding the Agent with $(T, P, E)$

- Never prompt vaguely (e.g. *"Build an ML model for my data"*). Agents default to simplistic, naive code.
- Use the formal **Task Formulation $(T, P, E)$** framework from Lecture 00:

| Component | What to Supply to the Agent |
| :--- | :--- |
| **Task ($T$)** | Objective (e.g. *Binary classification of loan default risk*), feature dictionary, and target variable name. |
| **Performance ($P$)** | Mandatory evaluation metrics (e.g. *Stratified 5-fold CV optimizing ROC-AUC and F1-score due to 5% class imbalance*). |
| **Experience ($E$)** | Training dataset characteristics ($N=5,000$ rows, $D=24$ features, missing values in column `age`). |
| **Architectural Rules** | Require leak-free `sklearn.pipeline.Pipeline`, explicit random seeds (`random_state=42`), and Weights & Biases logging. |


### The 4-Phase Prompting Strategy

- Break project development into modular, auditable steps:
  1. **Phase 1: Exploratory Data Analysis (EDA)**:
     - Prompt: *"Generate a Python script using pandas and seaborn to compute summary statistics, detect missing values, and plot correlations for dataset X."*
  2. **Phase 2: Leak-Free Preprocessing Pipeline**:
     - Prompt: *"Wrap StandardScaler and SimpleImputer inside an sklearn Pipeline. Ensure preprocessing fits only on training folds."*
  3. **Phase 3: Model Comparison & Tuning**:
     - Prompt: *"Compare Logistic Regression, Random Forest, and Gradient Boosting via StratifiedKFold cross-validation."*
  4. **Phase 4: Error Diagnostics & Interpretation**:
     - Prompt: *"Generate confusion matrices, ROC curves, and permutation feature importances for the best model."*


## Catching Agent Pitfall #1: Data Leakage

- **The Data Leakage Trap**: AI assistants frequently scale or impute data *before* splitting:

```python
# ❌ CRITICAL BUG (Common Agent Hallucination):
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # LEAKAGE: test distribution contaminates scaler!
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3)
```

- **Why it fails**: The mean $\mu$ and variance $\sigma^2$ of the test set leak into the training process, causing overly optimistic validation scores and deployment failure!
- **The Correct Defense**: Always enforce `sklearn.pipeline.Pipeline`:

```python
# ✅ LEAK-FREE PIPELINE:
pipe = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier())])
pipe.fit(X_train, y_train)  # Scaler learns ONLY from X_train!
```


## Catching Agent Pitfall #2: Metric Mismatch

- **The Accuracy Fallacy**: AI assistants routinely output `print("Accuracy:", accuracy_score(y_test, preds))` by default.
- On imbalanced tabular datasets (e.g. Fraud Detection with $1\%$ positive cases):
  - A naive dummy model predicting *"Never Fraud"* achieves **$99\%$ accuracy** while detecting $0$ fraudulent transactions!
- **Auditing Your Agent's Code**:
  - Check class proportions: `pd.Series(y).value_counts(normalize=True)`.
  - Enforce threshold-independent metrics: `roc_auc_score` and `average_precision_score` (PR-AUC).
  - Require balanced evaluation: Precision, Recall, and Macro $F_1$-score.
  - Verify stratification in splits: `train_test_split(..., stratify=y)` and `StratifiedKFold`.


In [ ]:
# Exemplary leak-free verification code
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Gold-standard leak-free pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Cross-validation strictly inside folds
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=StratifiedKFold(n_splits=5), scoring='roc_auc')
pipeline.fit(X_train, y_train)
test_auc = mt.roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])

print(f"5-Fold CV ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
print(f"Holdout Test ROC-AUC: {test_auc:.3f}")


## Oral Defensio Self-Check: 5 Essential Questions

- Before submitting your project, verify that you can answer these five questions without consulting an AI assistant:

1. **Mathematical Objective**: *What loss function is being minimized, and how do parameter updates take place?*
2. **Feature Preprocessing**: *Why was this specific scaler or encoder chosen, and where in the code is data leakage prevented?*
3. **Hyperparameter Rationale**: *Why were these specific values (e.g. `n_estimators=100`, `max_depth=4`, `C=1.0`) selected? What was the validation curve?*
4. **Metric Justification**: *Why is your chosen evaluation metric suitable for your domain task and class distribution?*
5. **Failure Analysis**: *When and why does your model make errors? Which instances in the confusion matrix represent costly mistakes?*

> 💡 **Key Takeaway:** AI coding agents are productivity multipliers for engineers who understand machine learning principles. Treat the agent as an eager junior assistant: verify every derivation, test every pipeline, and take personal ownership of the code!
